# Etapa 3: Transformación, Limpieza e Imputación en Capa Silver

Este cuaderno implementa las operaciones de limpieza analítica, tipado estricto, normalización de strings, imputaciones de valores faltantes (edad y renta) usando la mediana y el cálculoMonth-over-Month de altas de productos financieros mediante DuckDB.

In [1]:
import sys
import os
from pathlib import Path

# Resolver la ruta raíz del proyecto de forma dinámica y portable
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyarrow", "pandas", "scikit-learn", "joblib", "numpy"])
    print("Dependencias instaladas en Colab.")
    
    # Intentar montar Google Drive automáticamente si no está montado
    if not Path('/content/drive').exists():
        try:
            from google.colab import drive
            drive.mount('/content/drive')
        except Exception as e:
            print("No se pudo montar Drive automáticamente. Por favor, móntelo en el panel izquierdo de Colab.")

current_dir = Path(os.getcwd()).resolve()
if IN_COLAB:
    # Rutas de búsqueda comunes en Google Drive y Colab
    possible_paths = [
        Path('/content/drive/MyDrive/Colab Notebooks/Proyecto'),
        Path('/content/drive/MyDrive/Proyecto'),
        Path('/content/Proyecto'),
        Path('/content')
    ]
    for p in possible_paths:
        if (p / "notebooks").exists():
            current_dir = p / "notebooks"
            break

if current_dir.name == "notebooks":
    PROJECT_ROOT = current_dir.parent
else:
    PROJECT_ROOT = current_dir

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

## 1. Ejecutar Transformaciones de Capa Silver
Lanzamos el procesamiento relacional de la capa Silver.

In [2]:
from src.data_processing import DataProcessor

processor = DataProcessor(PROJECT_ROOT)
processor.bronze_to_silver()

2026-06-15 20:05:03,993 - INFO - [Silver] Iniciando transformaciones de limpieza y tipado...


2026-06-15 20:05:04,021 - INFO - [Silver] Procesando provincias...


2026-06-15 20:05:04,027 - INFO - [Silver] Procesando segmentos...


2026-06-15 20:05:04,033 - INFO - [Silver] Procesando productos...


2026-06-15 20:05:04,039 - INFO - [Silver] Procesando clientes...


2026-06-15 20:05:04,057 - INFO - [Silver] Procesando cliente_estado_mensual...


2026-06-15 20:05:04,069 - INFO - [Silver] Procesando cliente_producto_mensual...


2026-06-15 20:05:04,088 - INFO - [Silver] Calculando cliente_producto_alta...


2026-06-15 20:05:04,099 - INFO - [Silver] Procesamiento y exportación finalizados con éxito. Filas Silver: 2000 en 0.11s


## 2. Inspección del Impacto de Imputaciones en Calidad de Datos

In [3]:
import duckdb
import pandas as pd

con = duckdb.connect(database=":memory:")
silver_dir = PROJECT_ROOT / "data" / "silver"
bronze_dir = PROJECT_ROOT / "data" / "bronze"

# Registrar tablas de ambas capas para comparación
con.execute(f"CREATE VIEW bronze_estado AS SELECT * FROM read_parquet('{bronze_dir / 'cliente_estado_mensual.parquet'}')")
con.execute(f"CREATE VIEW silver_estado AS SELECT * FROM read_parquet('{silver_dir / 'cliente_estado_mensual.parquet'}')")

# Comparación de nulos en variables clave
nulls_comparison = con.execute("""
    SELECT 
        (SELECT COUNT(*) - COUNT(age) FROM bronze_estado) as nulos_edad_bronze,
        (SELECT COUNT(*) - COUNT(age) FROM silver_estado) as nulos_edad_silver,
        (SELECT COUNT(*) - COUNT(renta) FROM bronze_estado) as nulos_renta_bronze,
        (SELECT COUNT(*) - COUNT(renta) FROM silver_estado) as nulos_renta_silver
""").df()

print("Métricas de Calidad de Datos (Nulos):")
print(nulls_comparison)

Métricas de Calidad de Datos (Nulos):
   nulos_edad_bronze  nulos_edad_silver  nulos_renta_bronze  \
0                  1                  0                 455   

   nulos_renta_silver  
0                   0  
